<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/04-mcp-and-the-tool-ecosystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP & the Tool Ecosystem (concept)

**Goal:** Understand the Model Context Protocol — what it standardizes, when an FDE reaches for it, and how it relates to the raw tool-calling loop you built by hand.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **This is a concept notebook — no code to run.** The repo's thesis is *raw APIs, patterns not wrappers*; MCP is a protocol/wrapper layer, so we cover what it is and when it earns its place, not a from-scratch MCP server. You built the tool loop by hand in `01`–`03`; this is where that loop meets the wider ecosystem.

## The problem MCP solves

In `02-tool-design` you wrote each tool as a Python function plus a JSON schema, wired directly into *your* agent loop. That's perfect for learning and fine for one app. But scale it out and a painful M×N problem appears:

- You have **M AI apps** (a chatbot, an IDE assistant, an internal ops tool).
- You want them to reach **N systems** (GitHub, Postgres, Google Drive, your ticketing system).
- Naively, that's **M×N** bespoke integrations — every app re-implements every tool, in its own way.

**MCP (Model Context Protocol)** is an open standard, introduced by Anthropic and now broadly adopted, that turns M×N into M+N. Each system exposes its capabilities *once* as an **MCP server**; each app is an **MCP client** that can talk to *any* server. The official analogy: **MCP is USB-C for AI apps** — one standard plug instead of a custom cable per device.

## How it maps to the loop you already built

MCP doesn't replace anything you learned — it *standardizes the packaging* of the same three things a tool needs. Everything you built by hand has a named slot in the protocol:

| You built by hand (nb 01–03) | MCP's name for it | What it standardizes |
|---|---|---|
| A tool = function + JSON schema | **Tool** (on a server) | The tool definition the model sees |
| Your `HANDLERS`/`IMPLS` dispatch | The **server** runs it | Where execution happens — now out-of-process |
| Chunks/docs you stuffed in the prompt | **Resource** | Read-only context a server can expose |
| A reusable prompt template | **Prompt** | Named, shareable prompt scaffolds |
| Your agent loop calling the API | **Client** (inside the host app) | How the app discovers and calls servers |

The mechanics you learned are exactly what runs *underneath* MCP: the model still emits a tool-call request, something executes it, the result goes back. MCP just says "here's a standard way to describe and transport that, so a tool written once works in Claude Desktop, VS Code, Cursor, ChatGPT, and your own app without rewriting.

## When an FDE reaches for MCP (and when not)

**Reach for it when:**
- You're integrating with tools that **already have MCP servers** (GitHub, Slack, Postgres, filesystems — a large and growing catalog). Free integration beats writing your own.
- You want tools usable across **multiple host apps** — build the server once, plug it into Claude, Cursor, your app.
- You're inside an **MCP-native host** (Claude Desktop/Code, VS Code) and want to extend it.

**Skip it (keep the raw loop) when:**
- You have **a handful of app-specific tools** in one service — the M×N problem doesn't exist at M=1, and a direct function + schema is simpler than running a server.
- You need **tight control** over execution, latency, or the exact tool contract — indirection has a cost.
- You're **learning** — which is why this whole repo builds the loop by hand first. You can't evaluate whether MCP's abstraction helps until you know what it's abstracting.

The judgment, same as every framework question in this repo: **know the raw pattern, adopt the standard when the integration surface — not the demo — justifies it.**

## Security note — this is now a bigger attack surface

The moment your agent can reach arbitrary MCP servers — some written by third parties — the tools themselves become untrusted input. A malicious or compromised server can return content crafted to hijack your model (indirect prompt injection) or induce over-privileged actions (excessive agency). Those are exactly the risks the next section (`07-security`) covers. MCP makes the tool ecosystem bigger and more powerful, which makes the trust-boundary discipline more important, not less.

**Further reading:** the [MCP docs](https://modelcontextprotocol.io/) (spec, server/client guides, the public server catalog), and this repo's systems-level companion, the [Agent Orchestration walkthrough](https://www.calm.rocks/resources/prepare-interview/system-design/agent-orchestration-walkthrough/).

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Learn the raw tool loop first; adopt MCP when the *integration surface* justifies it | Reach for the protocol before you understand the loop it wraps |
| Use existing MCP servers (GitHub, Slack, Postgres…) — free integration | Re-implement every integration per app (the M×N trap) |
| Treat third-party MCP server output as untrusted input (see section 07) | Trust a public server's returned content because "it's a tool" |
| Scope each server's privileges tightly; human-gate consequential actions | Wire an agent to arbitrary servers with broad permissions |
| Keep the raw function+schema for a handful of app-specific tools | Stand up an MCP server for one tool in one service |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises (no code — reasoning)

1. **M×N in your own terms.** Name 3 AI apps and 3 backend systems from your work. How many bespoke integrations without MCP? With it? Where's the crossover point that makes the standard worth it?
2. **Draw the mapping.** Take one tool from `02-tool-design` and describe how it'd be exposed as an MCP *server* — what's the Tool, what (if anything) is a Resource, who's the client?
3. **Make the call.** For your capstone project (section 10): would you use MCP or a raw tool loop? Write the two-sentence justification you'd give an interviewer — the reasoning matters more than the answer.
4. **Threat-model a public server.** You add a third-party MCP server for "web search." List two ways its returned content could attack your agent, and how you'd mitigate each. (Revisit after `07-security`.)